In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
# ====================== 1. Import Required Libraries ======================
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# ====================== 2. Load Datasets ======================
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
test_ids = test['PassengerId']

# Combine train and test data for consistent preprocessing
all_data = pd.concat([train, test], ignore_index=True)

# ====================== 3. Feature Engineering ======================
# Split PassengerId into group ID and in-group number
all_data['Group'] = all_data['PassengerId'].apply(lambda x: x.split('_')[0])
all_data['GroupNum'] = all_data['PassengerId'].apply(lambda x: x.split('_')[1])

# Split Cabin into deck, cabin number, and side
cabin_split = all_data['Cabin'].str.split('/', expand=True)
all_data['Deck'] = cabin_split[0]
all_data['CabinNum'] = cabin_split[1]
all_data['Side'] = cabin_split[2]

# Drop useless columns
drop_cols = ['PassengerId', 'Cabin', 'Name']
all_data = all_data.drop(columns=drop_cols)

# ====================== 4. Missing Value Handling ======================
# Identify numeric columns
num_cols = all_data.select_dtypes(include=['number']).columns.tolist()
if 'Transported' in num_cols:
    num_cols.remove('Transported')

# Identify categorical and boolean columns
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
bool_cols = ['CryoSleep', 'VIP']

# Impute numeric features with median
imputer_num = SimpleImputer(strategy='median')
all_data[num_cols] = imputer_num.fit_transform(all_data[num_cols])

# Impute categorical features with mode
imputer_cat = SimpleImputer(strategy='most_frequent')
all_data[cat_cols] = imputer_cat.fit_transform(all_data[cat_cols])

# Impute boolean features with mode
for col in bool_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# ====================== 5. Encode Categorical Variables ======================
# Convert boolean features to 0/1
all_data['CryoSleep'] = all_data['CryoSleep'].astype(int)
all_data['VIP'] = all_data['VIP'].astype(int)

# Label encode other categorical columns
for col in cat_cols:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col])

# ====================== 6. Split Train and Test Sets ======================
train_processed = all_data[:len(train)]
test_processed = all_data[len(train):]

X = train_processed.drop('Transported', axis=1)
y = train_processed['Transported'].astype(int)
X_test = test_processed.drop('Transported', axis=1)

# Standardization (critical for MLP)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# ====================== Validation Strategy: 5-Fold Cross-Validation ======================
cv_folds = 5

# ====================== Baseline Model 1: Logistic Regression + Grid Search ======================
print("="*60)
print("Training Logistic Regression with Grid Search")
lr = LogisticRegression(max_iter=2000, random_state=42)
lr_params = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'saga']
}

grid_lr = GridSearchCV(lr, lr_params, cv=cv_folds, scoring='accuracy', n_jobs=1)
start = time.time()
grid_lr.fit(X_scaled, y)
end = time.time()

print(f"Best Hyperparameters: {grid_lr.best_params_}")
print(f"Cross-Validation Accuracy: {grid_lr.best_score_:.4f}")
print(f"Training Time: {end-start:.2f}s\n")

# ====================== Baseline Model 2: Random Forest + Random Search ======================
print("="*60)
print("Training Random Forest with Randomized Search")
rf = RandomForestClassifier(random_state=42)
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

rand_rf = RandomizedSearchCV(rf, rf_params, n_iter=6, cv=cv_folds, scoring='accuracy', random_state=42, n_jobs=1)
start = time.time()
rand_rf.fit(X_scaled, y)
end = time.time()

print(f"Best Hyperparameters: {rand_rf.best_params_}")
print(f"Cross-Validation Accuracy: {rand_rf.best_score_:.4f}")
print(f"Training Time: {end-start:.2f}s\n")

# ====================== Final Model: MLP Classifier (STABLE VERSION) ======================
print("="*60)
print("Training MLP Neural Network with Randomized Search")

mlp = MLPClassifier(
    max_iter=1500,
    random_state=42,
    early_stopping=True,
    tol=1e-4
)

mlp_params = {
    'hidden_layer_sizes': [(32,), (64,), (64,32)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.001, 0.01],
    'learning_rate': ['constant']
}

rand_mlp = RandomizedSearchCV(
    mlp, mlp_params,
    n_iter=5,
    cv=cv_folds,
    scoring='accuracy',
    random_state=42,
    n_jobs=1
)

start = time.time()
rand_mlp.fit(X_scaled, y)
end = time.time()

print(f"Best Hyperparameters: {rand_mlp.best_params_}")
print(f"Cross-Validation Accuracy: {rand_mlp.best_score_:.4f}")
print(f"Training Time: {end-start:.2f}s\n")

# ====================== Generate Final Kaggle Submission File ======================
best_model = rand_mlp.best_estimator_
test_pred = best_model.predict(X_test_scaled)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': test_pred.astype(bool)
})
submission.to_csv('submission_mlp_tuned.csv', index=False)

Training Logistic Regression with Grid Search
Best Hyperparameters: {'C': 1, 'solver': 'liblinear'}
Cross-Validation Accuracy: 0.7827
Training Time: 3.48s

Training Random Forest with Randomized Search
Best Hyperparameters: {'n_estimators': 100, 'min_samples_split': 5, 'max_depth': 20}
Cross-Validation Accuracy: 0.7441
Training Time: 26.16s

Training MLP Neural Network with Randomized Search
Best Hyperparameters: {'learning_rate': 'constant', 'hidden_layer_sizes': (32,), 'alpha': 0.001, 'activation': 'relu'}
Cross-Validation Accuracy: 0.7860
Training Time: 33.92s

